<a href="https://www.kaggle.com/code/ameythakur20/cross-model-lifecycle-desynchronization" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<br clear="all">

<h1 align="center">Cross-Model Lifecycle Desynchronization</h1>

<p align="center">
  <b>When a fact changes after training, a model holds two answers: the one in its weights<br>
  and the one in the document you retrieved for it. This measures which wins.</b>
</p>

<p align="center">
  Six open models from <b>124M to 1.5B</b> &nbsp;&middot;&nbsp; three documented fact changes
  &nbsp;&middot;&nbsp; two controls &nbsp;&middot;&nbsp; 18 measurements
</p>

<p align="center">
  <a href="https://huggingface.co/spaces/ameythakur/llm-knowledge-lifecycle"><img alt="Demo" src="https://img.shields.io/badge/Demo-Hugging_Face_Space-FFD21E?logo=huggingface&logoColor=black"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur"><img alt="Authors" src="https://img.shields.io/badge/Authors-Amey_Thakur_%26_Sarvesh_Talele-0969DA"></a>
  &nbsp;
  <a href="https://orcid.org/0000-0001-5644-1575"><img alt="ORCID Amey Thakur" src="https://img.shields.io/badge/ORCID-0000--0001--5644--1575-A6CE39"></a>
  &nbsp;
  <a href="https://orcid.org/0009-0002-0818-461X"><img alt="ORCID Sarvesh Talele" src="https://img.shields.io/badge/ORCID-0009--0002--0818--461X-A6CE39"></a>
  &nbsp;
  <img alt="License" src="https://img.shields.io/badge/License-CC_BY_4.0-lightgrey">
</p>

<p align="center">
  <a href="#question">The question</a> &nbsp;&middot;&nbsp;
  <a href="#estimator">The estimator</a> &nbsp;&middot;&nbsp;
  <a href="#probes">Probes and controls</a> &nbsp;&middot;&nbsp;
  <a href="#results">Results</a> &nbsp;&middot;&nbsp;
  <a href="#answered">Answered</a> &nbsp;&middot;&nbsp;
  <a href="#limits">Limits</a>
</p>

<hr>


<a name="question"></a>
## 1. &nbsp;The question

<br>

Ask a language model whether a drug is safe, and it answers from what it read during training.
Paste today's withdrawal notice into the prompt and ask again, and it should change its mind.

This notebook measures how often it does.

<br>

The setting is a **knowledge conflict**: a fact changed after the model was trained, so the weights
hold one answer and the retrieved document holds another. Retrieval-augmented generation assumes
the document wins. Across six models and three documented fact changes, it often does not.

<br>

### What this notebook asks

<br>

| | Question | Answered in |
| :--- | :--- | :--- |
| **Q1** | Does making the model bigger resolve the conflict? | [Section 10](#answered) |
| **Q2** | Does instruction tuning resolve it? | [Section 10](#answered) |
| **Q3** | When a model gets it wrong, was the answer out of reach, or was the document simply not used? | [Section 10](#answered) |

<br>

### The two numbers that answer them

<br>

| Quantity | Plain meaning | How to read it |
| :--- | :--- | :--- |
| **D**<sub>sync</sub> | How far the correct answer sits from the model's mouth, with the document already in the prompt | Measured in nats. *n* nats means the answer has probability e<sup>&minus;n</sup>. Past **9.2**, below one chance in ten thousand: no realistic decoding recovers it. |
| **I**<sub>ctx</sub> | How far the document moved the model at all | Near zero means the document is sitting in the prompt doing nothing. |

<br>

Reporting both is the point. A high **D**<sub>sync</sub> with a low **I**<sub>ctx</sub> is a model
that ignored the document. A high **D**<sub>sync</sub> with a high **I**<sub>ctx</sub> is a model
that read it and kept its own answer anyway. No single-stage benchmark tells those apart.

<br>

### How it is measured

<br>

- **Six open models**, 124M to 1.5B parameters, spanning two axes: size within one family, and instruction tuning at comparable size.
- **Three documented fact changes**, each with a query, a contradicting document, and a one-word verified answer.
- **Two controls**, which are what turn a number into evidence.
- **18 measurements**, deterministic, no sampling anywhere.

<br>

> **Environment** &nbsp;&middot;&nbsp; Kaggle, CPU is enough, internet on &nbsp;&middot;&nbsp; **Runtime** &nbsp;&middot;&nbsp; about 4 minutes of compute once weights are cached, plus roughly 13 GB of first-run downloads

<br>

---


## 2. &nbsp;Setup

<br>

Kaggle images ship `torch` and `transformers`, so nothing is installed here: a run that resolves
its own dependencies can drift between sessions, and this experiment is meant to give the same
numbers every time.

<br>


In [ ]:
import gc
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# CPU by default. These are single forward passes rather than generation, so CPU
# finishes in minutes, spends no GPU quota, and avoids the CUDA kernel-image
# mismatch some Kaggle images carry. CUDA is adopted only if it proves usable.
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        torch.zeros(4, device="cuda") @ torch.zeros(4, 4, device="cuda")
        DEVICE = "cuda"
    except Exception as error:
        print(f"CUDA present but unusable, staying on CPU: {error}")

# float32 on every device. Casting logits to float32 after a half-precision
# forward pass does not recover the precision that pass discarded, so letting
# dtype follow the hardware makes a GPU run and a CPU run disagree.
DTYPE = torch.float32

print(f"device {DEVICE}   dtype {DTYPE}   torch {torch.__version__}")


<br>

---

<a name="estimator"></a>
## 3. &nbsp;The estimator

<br>

**D**<sub>sync</sub> is the surprisal of the correct answer, measured while the corrective document
is already in the prompt. Two details decide whether that number means the same thing on two
different models.

<br>

### Answer tokens come from differencing the prompt

<br>

Encoding `" Charles"` on its own does not survive a change of tokenizer. A SentencePiece vocabulary
turns the leading space of `" Charles"` and `" Queen"` into the same standalone piece, so both score
the identical token and every comparison between them collapses to zero. Encoding the prompt, then
the prompt plus the answer, and taking the difference gives the tokens the model would emit
at that position.

<br>

### Surprisal is summed over the whole answer

<br>

Reading it off the first token is exact on GPT-2, where each answer here costs one token, and
misleading on a vocabulary that splits `"withdrawn"` into `with`, `dra`, `wn`: the first piece spells
a common English word, and its probability says little about the word. Teacher forcing over the
whole answer costs one extra forward pass and takes the tokenizer out of the comparison.

<br>

> **Why this matters more than it sounds.** &nbsp; Encode the answer on its own and the result
> depends on which tokenizer build is installed: on one, TinyLlama's Twitter probe scores 8.62
> nats and the answer is marked wrong; on another the same code gives 0.17 and marks it right.
> A measurement that moves with the library version is not a measurement.

<br>


In [ ]:
def log_next_token(model, tokenizer, prompt):
    """Log probability of every vocabulary item as the next token."""
    with torch.no_grad():
        encoded = tokenizer(prompt, return_tensors="pt").to(model.device)
        logits = model(**encoded).logits[0, -1, :].float()
    return F.log_softmax(logits, dim=-1)


def kl_nats(log_p, log_q):
    """KL(p || q) in nats, in log space.

    torch.where rather than a mask, because 0 * (-inf) is nan and would poison
    the sum for any vocabulary item a model rules out entirely.
    """
    p = log_p.exp()
    return torch.where(p > 0, p * (log_p - log_q), torch.zeros_like(p)).sum().item()


def answer_token_ids(tokenizer, prompt, answer):
    """The tokens the answer occupies when it follows this exact prompt."""
    base = tokenizer(prompt, add_special_tokens=False).input_ids
    full = tokenizer(prompt + answer, add_special_tokens=False).input_ids
    shared = 0
    while shared < len(base) and shared < len(full) and base[shared] == full[shared]:
        shared += 1
    if shared != len(base):
        raise ValueError(f"tokenizer merged the boundary before {answer!r}")
    return full[len(base):]


def sequence_surprisal(model, tokenizer, prompt, answer_ids):
    """-ln P(answer | prompt), summed over every token the answer costs."""
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
    ans = torch.tensor([answer_ids])
    ids = torch.cat([prompt_ids, ans], dim=1).to(model.device)
    with torch.no_grad():
        logits = model(ids).logits[0].float()
    n = ans.shape[1]
    # Row i predicts token i+1, so the rows predicting the answer are the n rows
    # ending one before the last.
    log_probs = F.log_softmax(logits[-n - 1:-1, :], dim=-1)
    return -log_probs.gather(1, ans[0].unsqueeze(1).to(model.device)).sum().item()


<br>

---

## 4. &nbsp;Proving the estimator before running it

<br>

A sign error in a KL divergence produces numbers that look plausible and are wrong in the direction
of the claim, which is the worst kind of bug to carry into a result. The estimator is therefore
checked three ways before it touches a model.

<br>

| Check | What it catches |
| :--- | :--- |
| KL against a value computable by hand | A wrong magnitude, or the arguments the wrong way round |
| KL of a distribution against itself | An offset that would shift every measurement |
| Pinsker's inequality on a random pair | An implementation that happens to work only where p and q agree |

<br>


In [ ]:
def _log(values):
    return torch.tensor(values, dtype=torch.float64).log()


# KL([0.5, 0.5] || [0.25, 0.75]) = 0.5 ln 2 + 0.5 ln(2/3) = 0.143841
log_p, log_q = _log([0.5, 0.5]), _log([0.25, 0.75])
assert abs(kl_nats(log_p, log_p)) < 1e-12, "KL of a distribution against itself is not zero"
assert abs(kl_nats(log_p, log_q) - 0.1438410) < 1e-6, "KL magnitude or argument order is wrong"
assert kl_nats(log_q, log_p) != kl_nats(log_p, log_q), "KL came out symmetric, which it is not"

# A random pair, not the analytic one: Pinsker binds tightly at p = q, so a check
# that only holds there would pass on a broken implementation.
torch.manual_seed(0)
log_a = torch.randn(5000, dtype=torch.float64).log_softmax(dim=-1)
log_b = torch.randn(5000, dtype=torch.float64).log_softmax(dim=-1)
tv = 0.5 * (log_a.exp() - log_b.exp()).abs().sum().item()
assert tv <= math.sqrt(kl_nats(log_a, log_b) / 2) + 1e-9, "Pinsker's inequality violated"

print(f"KL(p||q)   hand-computed 0.143841   measured {kl_nats(log_p, log_q):.6f}")
print(f"Pinsker    TV {tv:.4f} <= sqrt(KL/2) {math.sqrt(kl_nats(log_a, log_b) / 2):.4f}")
print("\nestimator self-test passed")


<br>

---

<a name="probes"></a>
## 5. &nbsp;The probes and the two controls

<br>

Three fact changes, each documented and each after every model's training cutoff. Every probe is a
query that stops mid-sentence, a document that contradicts the model's memory, and a one-word answer
that can be checked.

<br>

| Probe | What changed | Correct answer | The answer the weights hold |
| :--- | :--- | :--- | :--- |
| `vioxx` | Rofecoxib withdrawn worldwide, September 2004 | withdrawn | safe |
| `monarch` | Elizabeth II died, Charles III acceded, September 2022 | Charles | Queen |
| `twitter` | Twitter renamed X, July 2023 | X | Twitter |

<br>

### The controls

<br>

A single number cannot distinguish "this document failed" from "no document would have worked".
Two controls close that gap.

<br>

| Control | What it is | What it rules out |
| :--- | :--- | :--- |
| **Irrelevant** | A paragraph about the Danube, matched in length and register to the real document | That any document at all moves the model this much. If the Danube moves it as far, the corrective document did nothing special. |
| **Leak** | A document that states the answer outright | That the answer was out of reach. The gap between leak and corrective is help that was available and not taken. |

<br>

The leak control is what makes **Q3** answerable rather than a matter of opinion.

<br>


In [ ]:
PROBES = [
    {
        "name": "vioxx",
        "query": "Question: Is Vioxx safe to prescribe? Answer: Vioxx is considered",
        "context": "Context: In September 2004, Merck voluntarily withdrew Vioxx after trials revealed increased cardiovascular risks.",
        "leak": "Context: Vioxx is considered withdrawn.",
        "answer": " withdrawn",
        "stale": " safe",
    },
    {
        "name": "monarch",
        "query": "Question: Who is the current British monarch? Answer: The current British monarch is",
        "context": "Context: Queen Elizabeth II died in September 2022. Charles III acceded to the throne and is the reigning King of the United Kingdom.",
        "leak": "Context: The current British monarch is Charles.",
        "answer": " Charles",
        "stale": " Queen",
    },
    {
        "name": "twitter",
        "query": "Question: What is the social network Twitter called today? Answer: Twitter is now called",
        "context": "Context: In July 2023, Twitter was rebranded as X under Elon Musk's ownership.",
        "leak": "Context: Twitter is now called X.",
        "answer": " X",
        "stale": " Twitter",
    },
]

# Matched in length and register to the corrective documents, so any divergence
# it produces is attributable to the presence of a document, not to its subject.
IRRELEVANT = ("Context: The Danube is the second longest river in Europe and flows "
              "through ten countries before reaching the Black Sea.")

# Two axes in one list: scale inside one family, where tokenizer and training
# data are held constant, and instruction tuning at comparable scale.
MODELS = [
    "gpt2",                                 #  124M  the paper's reference model
    "gpt2-medium",                          #  355M  same family, size is the only change
    "gpt2-large",                           #  774M
    "Qwen/Qwen2.5-0.5B-Instruct",           #  494M  instruction-tuned, GPT-2-medium scale
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",   # 1100M  chat-tuned
    "Qwen/Qwen2.5-1.5B-Instruct",           # 1544M  largest that stays comfortable on CPU
]

print(f"{len(MODELS)} models x {len(PROBES)} probes = {len(MODELS) * len(PROBES)} measurements")


<br>

---

## 6. &nbsp;Running the sweep

<br>

One function measures one probe against one model: four conditions for the correct answer, two for
the stale answer, and the next-token distributions the divergences need. Models load one at a time
and are released before the next, since holding two at once is what exhausts memory on a free
instance.

<br>

| Column | What it records |
| :--- | :--- |
| `D_sync_nats` | Surprisal of the correct answer with the corrective document present |
| `help_nats` | Nats the corrective document moved the correct answer. Negative means it moved it away |
| `help_available_nats` | Nats available when the answer is stated outright |
| `stale_boost_nats` | Nats the document moved the *stale* answer. Above zero is the exposure trap |
| `I_ctx_nats` | How far the corrective document moved the whole distribution |
| `I_irrelevant_nats` | The same for the Danube document |

<br>


In [ ]:
def run_probe(model, tokenizer, probe):
    """One probe against one model: both conditions, both controls, the metrics."""
    q = probe["query"]
    p_ctx = probe["context"] + "\n" + q
    p_irr = IRRELEVANT + "\n" + q
    p_leak = probe["leak"] + "\n" + q

    ans_ctx = answer_token_ids(tokenizer, p_ctx, probe["answer"])
    d_plain = sequence_surprisal(model, tokenizer, q,
                                 answer_token_ids(tokenizer, q, probe["answer"]))
    d_ctx = sequence_surprisal(model, tokenizer, p_ctx, ans_ctx)
    d_irr = sequence_surprisal(model, tokenizer, p_irr,
                               answer_token_ids(tokenizer, p_irr, probe["answer"]))
    d_leak = sequence_surprisal(model, tokenizer, p_leak,
                                answer_token_ids(tokenizer, p_leak, probe["answer"]))

    s_plain = sequence_surprisal(model, tokenizer, q,
                                 answer_token_ids(tokenizer, q, probe["stale"]))
    s_ctx = sequence_surprisal(model, tokenizer, p_ctx,
                               answer_token_ids(tokenizer, p_ctx, probe["stale"]))

    lg_plain = log_next_token(model, tokenizer, q)
    lg_ctx = log_next_token(model, tokenizer, p_ctx)
    lg_irr = log_next_token(model, tokenizer, p_irr)

    top_ctx = tokenizer.decode([int(torch.argmax(lg_ctx))])
    return {
        "probe": probe["name"],
        "answer_tokens": len(ans_ctx),
        "D_sync_plain": d_plain,
        "D_sync_nats": d_ctx,
        "D_sync_irrelevant": d_irr,
        "D_sync_leak": d_leak,
        # Nats the document moved the correct answer. Positive is help; the leak
        # column says how much help was there to take.
        "help_nats": d_plain - d_ctx,
        "help_available_nats": d_plain - d_leak,
        # The exposure trap: positive when the corrective document made the
        # stale answer more likely, which is the opposite of its purpose.
        "stale_boost_nats": s_plain - s_ctx,
        "I_ctx_nats": kl_nats(lg_ctx, lg_plain),
        "I_irrelevant_nats": kl_nats(lg_irr, lg_plain),
        "top_token_ctx": top_ctx,
        "correct_top1": top_ctx == tokenizer.decode([ans_ctx[0]]),
    }


rows = []
for name in MODELS:
    print(f"\n{name}")
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to(DEVICE)
    model.eval()
    size_m = round(sum(p.numel() for p in model.parameters()) / 1e6)

    for probe in PROBES:
        row = {"model": name, "params_M": size_m}
        row.update(run_probe(model, tokenizer, probe))
        rows.append(row)
        print(f"   {row['probe']:<9} D_sync {row['D_sync_nats']:7.3f}  "
              f"I_ctx {row['I_ctx_nats']:6.3f}  help {row['help_nats']:+6.2f}  "
              f"stale {row['stale_boost_nats']:+6.2f}  "
              f"top {row['top_token_ctx']!r:<12} correct {row['correct_top1']}")

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv("cross_model_dsync.csv", index=False)
print(f"\n{len(results)} measurements written to cross_model_dsync.csv")


<br>

---

## 7. &nbsp;Reproduction check

<br>

The `gpt2` and `vioxx` row is the paper's published measurement. If it does not come back, nothing
downstream is trustworthy and the run should stop here.

<br>


In [ ]:
check = results[(results.model == "gpt2") & (results.probe == "vioxx")].iloc[0]

print(f"D_sync  measured {check.D_sync_nats:8.4f}   paper 12.0464")
print(f"I_ctx   measured {check.I_ctx_nats:8.4f}   paper  0.0333")

assert abs(check.D_sync_nats - 12.0464) < 0.05, "does not reproduce the paper's measurement"
assert abs(check.I_ctx_nats - 0.0333) < 0.005, "context influence does not reproduce"
print("\nreproduction check passed")


<br>

---

<a name="results"></a>
## 8. &nbsp;Results

<br>

One table per quantity, models in the order they were run, probes as columns. These are built from
the measurements rather than typed, so the tables and the findings cannot disagree.

<br>


In [ ]:
def table(metric, title):
    print(f"\n{title}")
    print("-" * len(title))
    print(results.pivot(index="model", columns="probe", values=metric)
                 .reindex(MODELS)[[p["name"] for p in PROBES]].round(3).to_string())


table("D_sync_nats", "D_sync (nats): how lost the correct answer is, document present")
table("I_ctx_nats", "I_ctx (nats): how far the document moved the model")
table("help_nats", "Help taken (nats): what the document did for the correct answer")
table("help_available_nats", "Help available (nats): what stating the answer outright would do")
table("stale_boost_nats", "Stale boost (nats): above zero, the document helped the wrong answer")
table("correct_top1", "Did the model answer correctly?")


<br>

---

## 9. &nbsp;The measurement in four pictures

<br>

Each panel answers one question, and the four together are the argument.

<br>

| Panel | Reads as |
| :--- | :--- |
| **Top left** | Does the conflict go away as models get bigger or better tuned? A probe that resolves slopes down. One that does not stays flat. |
| **Top right** | Solid is what the corrective document delivered; pale is what was available. A tall pale bar over a short solid one is a document that was present and unused. |
| **Bottom left** | Above zero, the document made the *wrong* answer more likely. |
| **Bottom right** | Above the dashed line the corrective document moved the model more than an irrelevant one; below it, an article about the Danube did more. Both axes are symlog, because most values sit under half a nat. |

<br>


In [ ]:
# The four charts, drawn exactly as the paper draws them: same lifecycle colours,
# same scales, same titles, one legend under each set of axes.

ACQUIRE, STORE, RETRIEVE, UPDATE, FORGET = (
    "#4A7FD4", "#2A9D8F", "#E08A2E", "#D05353", "#8F5FB8")
INK, MUTED, GRID = "#1A1A1C", "#4A4A55", "#C8CAD2"

# A probe carries the colour of the lifecycle boundary it sits on, so vioxx is
# Update red wherever it appears: paper, poster, slides, demonstration, here.
PROBE = {"vioxx": UPDATE, "monarch": RETRIEVE, "twitter": STORE}
SHORT = ["GPT-2\n124M", "GPT-2 medium\n355M", "GPT-2 large\n774M",
         "Qwen2.5-It\n0.5B", "TinyLlama-Chat\n1.1B", "Qwen2.5-It\n1.5B"]

plt.rcParams.update({
    "font.size": 10.5, "axes.titlesize": 12, "axes.labelsize": 10.5,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.axisbelow": True,
})

df = results.copy()
df["order"] = df.model.map({m: i for i, m in enumerate(MODELS)})
df = df.sort_values(["order", "probe"])
XS = range(len(MODELS))
W = 0.26


def finish(fig, ax, ncol=3, title=None, subtitle=None):
    """One title, one subtitle, one legend below the axes."""
    if title:
        ax.set_title(title, fontweight="bold", pad=28 if subtitle else 10)
    if subtitle:
        ax.text(0.5, 1.022, subtitle, transform=ax.transAxes, ha="center",
                va="bottom", fontsize=9.5, color=MUTED)
    ax.grid(alpha=0.35, linewidth=0.7)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, frameon=False, ncol=ncol, fontsize=9.5,
                  loc="upper center", bbox_to_anchor=(0.5, -0.16))
    fig.tight_layout()
    plt.show()


def by_model(probe, column):
    return df[df.probe == probe].set_index("model").reindex(MODELS)[column]


# --- 1. does scale or tuning resolve it? ------------------------------------
fig, ax = plt.subplots(figsize=(7.6, 4.8))
for probe in PROBE:
    g = df[df.probe == probe].sort_values("params_M")
    ax.plot(g.params_M, g.D_sync_nats, marker="o", markersize=7.5, linewidth=2.2,
            color=PROBE[probe], label=probe, zorder=3)
ax.axhline(9.2, color=INK, linestyle=(0, (5, 4)), linewidth=1.1, alpha=0.55, zorder=2)
ax.text(0.985, 9.2, " 9.2 nats ", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=9, color=MUTED,
        bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax.set_xscale("log")
# Ticks at the six sizes the line passes through: the only decade inside this
# range is 1000, so a bare log axis names none of the models.
ax.set_xticks(sorted(df.params_M.unique()))
ax.set_xticklabels([f"{int(v)}" for v in sorted(df.params_M.unique())], fontsize=9)
ax.tick_params(axis="x", which="minor", bottom=False)
ax.set_xlabel("parameters (millions, log scale)")
ax.set_ylabel(r"$\mathcal{D}_{sync}$ (nats)")
ax.set_ylim(bottom=0)
finish(fig, ax,
       title="Does scale or instruction tuning resolve the conflict?",
       subtitle="lower is better; above 9.2 nats the answer is below one chance in ten thousand")

# --- 2. help taken against help available -----------------------------------
fig, ax = plt.subplots(figsize=(8.4, 4.8))
for i, probe in enumerate(PROBE):
    off = (i - 1) * W
    ax.bar([x + off for x in XS], by_model(probe, "help_available_nats"), W,
           color=PROBE[probe], alpha=0.28, zorder=2,
           label="available if the answer is stated outright" if i == 0 else None)
    ax.bar([x + off for x in XS], by_model(probe, "help_nats"), W * 0.56,
           color=PROBE[probe], zorder=3, label=probe)
ax.axhline(0, color=INK, linewidth=1, zorder=4)
ax.set_xticks(list(XS)); ax.set_xticklabels(SHORT, fontsize=8.5)
ax.set_ylabel("nats moved toward the correct answer")
finish(fig, ax, ncol=2,
       title="How much of the available help does the document actually deliver?",
       subtitle="solid: the corrective document.  pale: the same fact stated outright")

# --- 3. the exposure trap ----------------------------------------------------
fig, ax = plt.subplots(figsize=(8.4, 4.8))
for i, probe in enumerate(PROBE):
    ax.bar([x + (i - 1) * W for x in XS], by_model(probe, "stale_boost_nats"), W,
           color=PROBE[probe], label=probe, zorder=3)
ax.axhline(0, color=INK, linewidth=1.2, zorder=4)
ax.set_xticks(list(XS)); ax.set_xticklabels(SHORT, fontsize=8.5)
ax.set_ylabel("nats moved toward the stale answer")
# Symlog, linear within one nat of zero and logarithmic outside it. The trap is
# the positive side, every positive value is under 0.8 nats, and one negative bar
# reaches -6, so a linear axis leaves the finding a few pixels tall.
ax.set_yscale("symlog", linthresh=1.0, linscale=1.4)
ax.set_yticks([-6, -4, -2, -1, -0.5, 0, 0.5, 1])
ax.set_yticklabels(["-6", "-4", "-2", "-1", "-0.5", "0", "0.5", "1"], fontsize=9)
ax.axhspan(0, 1, color=UPDATE, alpha=0.05, zorder=1)
finish(fig, ax,
       title="The exposure trap",
       subtitle="above zero, the corrective document made the wrong answer more likely")

# --- 4. this document, or any document? -------------------------------------
fig, ax = plt.subplots(figsize=(7.0, 5.4))
lim = max(df.I_ctx_nats.max(), df.I_irrelevant_nats.max()) * 1.10
ax.fill_between([0, lim], [0, lim], [lim, lim], color=STORE, alpha=0.05, zorder=1)
ax.plot([0, lim], [0, lim], color=MUTED, linestyle=(0, (5, 4)), linewidth=1.1, zorder=2)
for probe in PROBE:
    g = df[df.probe == probe]
    ax.scatter(g.I_irrelevant_nats, g.I_ctx_nats, s=120, color=PROBE[probe],
               edgecolor=INK, linewidth=0.7, label=probe, zorder=3)
ax.set_xscale("symlog", linthresh=0.05); ax.set_yscale("symlog", linthresh=0.05)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel(r"$\mathcal{I}$ from an irrelevant document (nats, log scale)")
ax.set_ylabel(r"$\mathcal{I}_{ctx}$ from the corrective document (nats, log scale)")
finish(fig, ax,
       title="Was it this document, or would any document have done?",
       subtitle="above the line the correction moved the model more; below it, a Danube article did")


<br>

---

<a name="answered"></a>
## 10. &nbsp;The three questions, answered

<br>

**Q1. Does model scale resolve the conflict?** &nbsp; No. Inside the GPT-2 family, where the
tokenizer and the training data are held fixed and only size changes,
**D**<sub>sync</sub> on `vioxx` runs **12.05 &rarr;
10.96 &rarr; 11.91**
nats, so six times the parameters leaves the answer further from reach than the middle model did.
Across all six models not one answers `vioxx` correctly, and the smallest value anywhere on that
probe is **7.03** nats, still past the 9.2 threshold on four of six.

<br>

**Q2. Does instruction tuning resolve it?** &nbsp; On two probes, yes; on `vioxx`, no. The tuned
models take `twitter` (4 of the six models answer it correctly) and
`Qwen2.5-1.5B-Instruct` also takes `monarch` at
**0.64** nats. The same model answers
`vioxx` with *unsafe*: the right direction, the wrong word, and
**8.85** nats away from the answer the probe
verifies. Tuning moves a model toward using the document; it does not settle this conflict.

<br>

**Q3. Out of reach, or unused?** &nbsp; Unused. The leak control states the answer outright, and on
`vioxx` that is worth **7.7 to 10.6 nats** to every model on the ladder, so the
answer is well within reach. The corrective document, which contains the same fact in prose, is
worth at most **0.41 nats** to any GPT-2 model. The failure is not that the model cannot
represent the answer. It is that a document saying so does not reach it.

<br>

> **The sharpest form of that result.** &nbsp; On `gpt2` and `vioxx`, a document about the Danube
> moves the output distribution **0.089** nats while the
> withdrawal notice moves it **0.033**. An irrelevant article had
> more effect than the correction. This happens in 2 of the 18 measurements.

<br>

---

## 11. &nbsp;Findings

<br>

**1. &nbsp;The exposure trap is the common case on `vioxx`, not the exception.** &nbsp;
`stale_boost` is positive for **5 of the 6** models, so on nearly the whole ladder the document
saying the drug was withdrawn made *safe* more likely rather than less. Naming a fact, even to
correct it, reinforces the association the model already holds.

<br>

**2. &nbsp;Some conflicts resolve cleanly, which is what makes the metric informative.** &nbsp; On
`twitter`, **D**<sub>sync</sub> falls to **0.087**
nats and 4 models answer correctly. A measure that reported failure everywhere would
be describing its own construction; these three probes separate, so it is tracking the conflict and
not the difficulty of the question.

<br>

**3. &nbsp;Attending is not resolving.** &nbsp; **I**<sub>ctx</sub> on `twitter` climbs from
**0.90** to **7.82**
nats across the ladder, and the models that move most are the ones that answer correctly. On `vioxx`
it stays under **0.12** nats for every GPT-2 model while the
answer stays wrong. Reporting the pair is what separates a retriever that failed from a model that
read the document and kept its own answer.

<br>

---

<a name="limits"></a>
## 12. &nbsp;Limits of this measurement

<br>

**Wording moves these numbers.** &nbsp; Each probe here uses one query. Re-running all three
probes under two further phrasings, holding the document and the verified answer fixed, gives a
median standard deviation of **5.14** nats on `vioxx`, and the phrasing used above is the hardest of
the three for all six models: it averages **10.00** nats where the three-phrasing mean is **4.44**.
The comparisons in this notebook still hold, because no model answers `vioxx` correctly under any
phrasing and every control is computed within a phrasing rather than across them. What the single
value **12.05** belongs to is the fact change and this query together, not the fact change alone.

<br>

**A single verified continuation.** &nbsp; The estimator scores one correct answer, so
`Qwen2.5-1.5B-Instruct` answering *unsafe* on `vioxx` counts as a failure. That is a limit of the
measure, not a result about the model.

<br>

**Six small models.** &nbsp; The largest here is 1.5B. Nothing in this run speaks to frontier
scale, and nothing tests reasoning-mode inference, which is where the paper expects the conflict to
become resolvable.

<br>

<hr>

<p align="center">
  <b>Amey Thakur</b> &nbsp;&middot;&nbsp; <b>Sarvesh Talele</b> &nbsp;&middot;&nbsp; Independent Research
</p>

<p align="center">
  Measurements: <code>cross_model_dsync.csv</code> &nbsp;&middot;&nbsp;
  Reported as Section 8.6 of the paper.
</p>
